In [22]:

# 1. Imports

import numpy as np
import re
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Input, Dropout


In [23]:
def sample_with_temperature(preds, temperature=0.8):
    preds = np.asarray(preds).astype("float64")
    preds = np.log(preds + 1e-8) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    return np.random.choice(len(preds), p=preds)


In [24]:

# 2. Load & preprocess dataset

with open(r"C:\Users\ASUS\project\harrypotter.txt", "r", encoding="utf-8") as f:
    text = f.read().lower()

# remove punctuation
text = re.sub(r"[^a-z\s]", "", text)

words = text.split()


In [25]:

# 3. Tokenization

vocab = sorted(set(words))
word_to_idx = {w: i for i, w in enumerate(vocab)}
idx_to_word = {i: w for w, i in word_to_idx.items()}
vocab_size = len(vocab)

encoded = [word_to_idx[word] for word in words]


In [26]:

# 4. Create input-output sequences

seq_length = 20
X, y = [], []

for i in range(len(encoded) - seq_length):
    X.append(encoded[i:i + seq_length])
    y.append(encoded[i + seq_length])

X = np.array(X)
y = np.array(y)


In [27]:

# 5. Train-validation split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [28]:

# 6. Build LSTM model

model = Sequential([
    Input(shape=(seq_length,)),
    Embedding(vocab_size, 128),
    LSTM(256, return_sequences=True),
    Dropout(0.3),
    LSTM(256),
    Dense(vocab_size, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy"
)


In [29]:

# 7. Train model

early_stop = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=64,
    callbacks=[early_stop]
)


Epoch 1/20
969/969 ━━━━━━━━━━━━━━━━━━━━ 100s 100ms/step - loss: 6.7672 - val_loss: 6.5102
Epoch 2/20
969/969 ━━━━━━━━━━━━━━━━━━━━ 94s 97ms/step - loss: 6.3254 - val_loss: 6.3832
Epoch 3/20
969/969 ━━━━━━━━━━━━━━━━━━━━ 95s 98ms/step - loss: 6.1769 - val_loss: 6.3747
Epoch 4/20
969/969 ━━━━━━━━━━━━━━━━━━━━ 94s 97ms/step - loss: 6.0137 - val_loss: 6.2698
Epoch 5/20
969/969 ━━━━━━━━━━━━━━━━━━━━ 94s 97ms/step - loss: 5.8074 - val_loss: 6.2081
Epoch 6/20
969/969 ━━━━━━━━━━━━━━━━━━━━ 94s 97ms/step - loss: 5.5994 - val_loss: 6.1575
Epoch 7/20
969/969 ━━━━━━━━━━━━━━━━━━━━ 94s 97ms/step - loss: 5.3888 - val_loss: 6.1549
Epoch 8/20
969/969 ━━━━━━━━━━━━━━━━━━━━ 95s 98ms/step - loss: 5.1825 - val_loss: 6.1851
Epoch 9/20
969/969 ━━━━━━━━━━━━━━━━━━━━ 95s 98ms/step - loss: 4.9939 - val_loss: 6.2354
Epoch 10/20
969/969 ━━━━━━━━━━━━━━━━━━━━ 95s 98ms/step - loss: 4.8170 - val_loss: 6.3249


In [30]:

# 8. Text generation function

def generate_text(seed_text, next_words=40, temperature=0.8):
    for _ in range(next_words):
        tokenized = [word_to_idx.get(w, 0) for w in seed_text.split()]
        tokenized = tokenized[-seq_length:]

        if len(tokenized) < seq_length:
            tokenized = [0] * (seq_length - len(tokenized)) + tokenized

        prediction = model.predict(np.array([tokenized]), verbose=0)[0]
        next_idx = sample_with_temperature(prediction, temperature)
        next_word = idx_to_word[next_idx]

        seed_text += " " + next_word

    return seed_text



In [31]:

# 9. Generate sample text

print(generate_text("harry", 40, temperature=0.8))



harry said uncle would be on the hands there had got to the minutes who was there he had smiled you one in the ghost so uncle dursley actually back to looked back harry mr vernon didnt know he could a


In [33]:

# 10. Save model

model.save("lstm_txt_gen_dg).keras")
